# Active Learning Experiment

Comparing entropy-based vs random sampling strategies.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt

## 1. Load Data and Run AL

In [ ]:
from agents.al_agent import ActiveLearningAgent

df = pd.read_parquet('../data/labeled/labeled.parquet')
print(f'Loaded: {len(df)} rows, classes: {df["label"].nunique()}')

agent = ActiveLearningAgent(model='logreg')
history = agent.run_cycle(
    labeled_df=df, pool_df=df,
    strategy='entropy', n_iterations=5, batch_size=20
)

## 2. Learning Curves: Entropy vs Random

In [ ]:
with open('../data/active/history_entropy.json') as f:
    he = json.load(f)
with open('../data/active/history_random.json') as f:
    hr = json.load(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.plot([h['n_labeled'] for h in he], [h['accuracy'] for h in he], 'o-', label='Entropy', lw=2)
ax1.plot([h['n_labeled'] for h in hr], [h['accuracy'] for h in hr], 's--', label='Random', lw=2)
ax1.set_title('Accuracy vs N Labeled')
ax1.set_xlabel('N labeled'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot([h['n_labeled'] for h in he], [h['f1'] for h in he], 'o-', label='Entropy', lw=2)
ax2.plot([h['n_labeled'] for h in hr], [h['f1'] for h in hr], 's--', label='Random', lw=2)
ax2.set_title('F1 vs N Labeled')
ax2.set_xlabel('N labeled'); ax2.set_ylabel('F1')
ax2.legend(); ax2.grid(alpha=0.3)

plt.suptitle('Active Learning: Entropy vs Random')
plt.tight_layout()
plt.show()

## 3. Sample Savings Analysis

In [ ]:
fe = he[-1]
fr = hr[-1]

print(f'Entropy — Accuracy: {fe["accuracy"]}, F1: {fe["f1"]}, Samples: {fe["n_labeled"]}')
print(f'Random  — Accuracy: {fr["accuracy"]}, F1: {fr["f1"]}, Samples: {fr["n_labeled"]}')

# How many samples does random need to reach entropy's F1?
target_f1 = fe['f1']
for step in hr:
    if step['f1'] >= target_f1:
        saved = step['n_labeled'] - fe['n_labeled']
        print(f'\nEntropy saved {max(0, saved)} samples vs random to reach F1={target_f1}')
        break
else:
    print(f'\nRandom never reached entropy\'s F1={target_f1} — entropy is strictly better')

## 4. Conclusion

Entropy-based Active Learning selects the most uncertain samples for labeling,
which typically leads to faster model improvement compared to random sampling.
This means we can achieve the same model quality with fewer labeled examples,
saving annotation time and cost.